In [1]:
import numpy as np
import matplotlib.pyplot as plt
import h5py

In [2]:
# ASTRID cosmology values
hubble, z = 0.6774, 2.5

# Calculating Px

As I will need to obtain the radial separation between two skewers, I cannot mask just yet. I need to do that afterwards. We will call a folder that contains all the ffts without any masking

In [4]:
data = '/pscratch/sd/l/lflores/astrid_hcd_outputs/FFTs/ffts_0.3_40/minibox_01.hdf5'
Nmbox = 25

with h5py.File(data, 'r') as f:
    print('Atributes:')
    for k in f.attrs.keys():
        print(f'{k} = {f.attrs[k]}')
    print('----------------') 
    # Atributes
    logNHi_min = f.attrs['logNHI_min']
    logNHi_max = f.attrs['logNHI_max']
    smth_factor = f.attrs['Smoothing factor']
    Lbox = f.attrs['box_size_Mpch']
    Lmbox = f.attrs['minibox_size_Mpch']
    Nsk = f.attrs['skewers_per_side']
    Np = f.attrs['pixels_per_skewer']
    Pw = f.attrs['pixel_width_Mpch']
    Ssk = f.attrs['skewer_separation_Mpch']
    print('Data:')
    print(f.keys())
    # Data
    fft_tot = f['fft_tot'][:]
    print('fft_tot shape:', fft_tot.shape)
    fft_lya = f['fft_lya'][:]
    print('fft_lya shape:', fft_lya.shape)
    fft_hcd = f['fft_hcd'][:]
    print('fft_hcd shape:', fft_hcd.shape)
    fft_lyahcd = f['fft_lyahcd'][:]
    print('fft_lyahcd shape:', fft_lyahcd.shape)
    k_los = f['k_los'][:]
    print('k_los shape', k_los.shape)
    C = f['C'][()]
    print('C = ', C)
    colden = f['colden'][:]
    print('colden shape', colden.shape)


Atributes:
Smoothing factor = 0.3
box_size_Mpch = 250
logNHI_max = 40.0
logNHI_min = 0
minibox_size_Mpch = 50.0
pixel_width_Mpch = 0.1
pixels_per_skewer = 2500
skewer_separation_Mpch = 0.5
skewers_per_side = 100
----------------
Data:
<KeysViewHDF5 ['C', 'colden', 'fft_hcd', 'fft_lya', 'fft_lyahcd', 'fft_tot', 'k_los']>
fft_tot shape: (10000, 1251)
fft_lya shape: (10000, 1251)
fft_hcd shape: (10000, 1251)
fft_lyahcd shape: (10000, 1251)
k_los shape (1251,)
C =  0.005908550712377059
colden shape (10000, 2500)


In [56]:
ix, iy = np.divmod(np.arange(Nsk), Lmbox)  # This gives me the coordinates of each skewer within the minibox
ix, iy

(array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]),
 array([ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12.,
        13., 14., 15., 16., 17., 18., 19., 20., 21., 22., 23., 24., 25.,
        26., 27., 28., 29., 30., 31., 32., 33., 34., 35., 36., 37., 38.,
        39., 40., 41., 42., 43., 44., 45., 46., 47., 48., 49.,  0.,  1.,
         2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12., 13., 14.,
        15., 16., 17., 18., 19., 20., 21., 22., 23., 24., 25., 26., 27.,
        28., 29., 30., 31., 32., 33., 34., 35., 36., 37., 38., 39., 40.,
        41., 42., 43., 44., 45., 46., 4

# Option 1: the division of the box is wrong

In [15]:
def div_box(Nbox, Nsk, Np, *arrays):
    """ This function divides an original array with shape (Nsk**2, Np) into Nbox miniboxes

    Parameters:
    -----------------
    Nbox : int
        Number of miniboxes. If the original shape is not a multiple of Nmbox, the functions gets an error.
    Nsk : int 
        Number of skewer per side in the original box..
    Np : int
        Number of pixel per skewer in the original box.
    arrays : n-array(s)
        Original array(s) that the user wants to divide.

    Returns:
    ----------------
    new_array : list
        Divided array(s) into desired shape
    """

    mb_per_side = int(np.sqrt(Nbox))  # Number of miniboxes per side
    if mb_per_side**2 != Nbox:
        raise ValueError("Nbox must be a perfect square")
    
    mb_size = Nsk // mb_per_side  # Number of skewers per side inside each minibox

    # Computing slices
    slices = []
    for j_mb in range(mb_per_side):  
        for i_mb in range(mb_per_side):
            imin, imax = i_mb*mb_size, (i_mb+1)*mb_size
            jmin, jmax = j_mb*mb_size, (j_mb+1)*mb_size
            slices.append((imin, imax, jmin, jmax))

    results = []
    for arr in arrays:
        
        if arr.shape != (Nsk**2, Np):
            raise ValueError(f"Expected shape {(Nsk**2, Np)}, got {arr.shape}")
            
        grid = arr.reshape(Nsk, Nsk, Np)
        new_array = np.zeros((Nbox, mb_size**2, Np))
        for mb_id, (imin, imax, jmin, jmax) in enumerate(slices):
            minibox = grid[imin:imax, jmin:jmax, :]
            new_array[mb_id] = minibox.reshape(-1, Np)

        results.append(new_array) 

    return results, mb_size

In [18]:
simple = np.zeros((10, 10, 100))
for i in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]:
    for j in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]:
        simple[i, j, :] = i + j*0.1
simple

array([[[0. , 0. , 0. , ..., 0. , 0. , 0. ],
        [0.1, 0.1, 0.1, ..., 0.1, 0.1, 0.1],
        [0.2, 0.2, 0.2, ..., 0.2, 0.2, 0.2],
        ...,
        [0.7, 0.7, 0.7, ..., 0.7, 0.7, 0.7],
        [0.8, 0.8, 0.8, ..., 0.8, 0.8, 0.8],
        [0.9, 0.9, 0.9, ..., 0.9, 0.9, 0.9]],

       [[1. , 1. , 1. , ..., 1. , 1. , 1. ],
        [1.1, 1.1, 1.1, ..., 1.1, 1.1, 1.1],
        [1.2, 1.2, 1.2, ..., 1.2, 1.2, 1.2],
        ...,
        [1.7, 1.7, 1.7, ..., 1.7, 1.7, 1.7],
        [1.8, 1.8, 1.8, ..., 1.8, 1.8, 1.8],
        [1.9, 1.9, 1.9, ..., 1.9, 1.9, 1.9]],

       [[2. , 2. , 2. , ..., 2. , 2. , 2. ],
        [2.1, 2.1, 2.1, ..., 2.1, 2.1, 2.1],
        [2.2, 2.2, 2.2, ..., 2.2, 2.2, 2.2],
        ...,
        [2.7, 2.7, 2.7, ..., 2.7, 2.7, 2.7],
        [2.8, 2.8, 2.8, ..., 2.8, 2.8, 2.8],
        [2.9, 2.9, 2.9, ..., 2.9, 2.9, 2.9]],

       ...,

       [[7. , 7. , 7. , ..., 7. , 7. , 7. ],
        [7.1, 7.1, 7.1, ..., 7.1, 7.1, 7.1],
        [7.2, 7.2, 7.2, ..., 7.2, 7.2, 7.2

In [32]:
divided, _ = div_box(4, 10, 100, simple.reshape(100, 100))

In [48]:
divided[0][0][6, :]

array([1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1,
       1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1,
       1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1,
       1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1,
       1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1,
       1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1,
       1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1,
       1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1, 1.1])

So this function works properly, good!

# Option 2: the divmod function does not do what I thought it did

In [55]:
divided[0][0]

array([[0. , 0. , 0. , ..., 0. , 0. , 0. ],
       [0.1, 0.1, 0.1, ..., 0.1, 0.1, 0.1],
       [0.2, 0.2, 0.2, ..., 0.2, 0.2, 0.2],
       ...,
       [4.2, 4.2, 4.2, ..., 4.2, 4.2, 4.2],
       [4.3, 4.3, 4.3, ..., 4.3, 4.3, 4.3],
       [4.4, 4.4, 4.4, ..., 4.4, 4.4, 4.4]], shape=(25, 100))

In [57]:
iy, ix = np.divmod(25, 250/2)  # This gives me the coordinates of each skewer within the minibox

In [59]:
iy, ix

(np.float64(0.0), np.float64(25.0))